# Coursera to Google Drive

Runs in Google Colab, downloads your enrolled Coursera courses, and saves them straight into your Google Drive. Nothing is installed on your PC.

**Before you start:** log in to coursera.org in your browser, press F12, open Application > Cookies > https://www.coursera.org, and copy the value of the `CAUTH` cookie.

Run the cells top to bottom (Shift+Enter).

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install the downloader
`coursera-helper` is the maintained fork of coursera-dl. `setuptools` supplies `distutils`, which newer Python removed and the tool still imports.

In [ ]:
!pip install -q coursera-helper setuptools
!coursera-helper --help | head -3

## 3. Settings

In [ ]:
#@title Settings { run: "auto" }
SAVE_TO = "/content/drive/MyDrive/Coursera"  #@param {type:"string"}
VIDEO_RESOLUTION = "720p"  #@param ["360p", "540p", "720p"]
SUBTITLES = "en"  #@param {type:"string"}
DOWNLOAD_NOTEBOOKS = True  #@param {type:"boolean"}
DOWNLOAD_QUIZZES = False  #@param {type:"boolean"}

import os, getpass
os.makedirs(SAVE_TO, exist_ok=True)
if not os.environ.get("CAUTH"):
    # Hidden input, so the cookie never gets saved into the notebook
    os.environ["CAUTH"] = getpass.getpass("Paste your Coursera CAUTH cookie: ").strip()
print("Saving to", SAVE_TO)

## 4. See which courses you can download
Copy the slugs you want from this list into the next cell.

In [ ]:
!coursera-helper --cauth "$CAUTH" --list-courses

## 5. Download
Put one course slug per line. Safe to rerun: files already in Drive are skipped, so if Colab disconnects just run cells 1, 3 and 5 again.

In [ ]:
COURSES = """
machine-learning
""".split()

import subprocess, shlex
extra = []
if DOWNLOAD_NOTEBOOKS: extra.append("--download-notebooks")
if DOWNLOAD_QUIZZES: extra.append("--download-quizzes")

for slug in COURSES:
    print(f"
===== {slug} =====", flush=True)
    cmd = ["coursera-helper", "--cauth", os.environ["CAUTH"],
           "--path", SAVE_TO,
           "--video-resolution", VIDEO_RESOLUTION,
           "--subtitle-language", SUBTITLES,
           "--resume", *extra, slug]
    subprocess.run(cmd)
print("
Done. Files are in", SAVE_TO)

## Troubleshooting

- **403 Forbidden:** your CAUTH cookie is wrong or expired. Log in again, copy a fresh one, run `os.environ.pop("CAUTH")` and rerun cell 3.
- **Colab disconnected:** free sessions last up to about 12 hours and drop after about 90 minutes idle. Keep the tab open, then rerun. Finished files are skipped.
- **A course downloads nothing:** some course types (Projects, some Specialization items) aren't supported by the tool.